In [23]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict
import os
from dotenv import load_dotenv

In [24]:
load_dotenv()


True

In [25]:
GROK_API_KEY = os.getenv("GROK_API_KEY")

llm = ChatOpenAI(
    model="qwen/qwen3.8-27b",
    temperature=0,
    api_key=GROK_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    # Groq free tier caps OUTPUT at 1000 tokens/minute (OTPM). Keep every
    # single response under that so a call isn't rejected before it runs.
    max_tokens=900,
)

In [26]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

In [27]:
def create_outline(state:BlogState)->BlogState:
    title=state['title']
    prompt=f'Generate a detailed outline for a blog on the topic -{title}'
    outline=llm.invoke(prompt).content
    state['outline']=outline
    return state

In [28]:
def create_blog(state:BlogState)->BlogState:
    title=state['title']
    outline=state['title']

    prompt=f'write a detailed blog on the title - {title} using the following outline {outline}'
    content=llm.invoke(prompt).content
    state['content']=content
    return state

In [29]:
graph=StateGraph(BlogState)
graph.add_node('create_outline',create_outline)

graph.add_node('create_blog',create_blog)


graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')

graph.add_edge('create_blog',END)

workflow=graph.compile()

In [30]:
initial_state={'title':'Rise of tech from web  1.0 to AI'}
final_state=workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of tech from web  1.0 to AI', 'outline': 'Here is a detailed outline for a blog post titled **"From Static Pages to Neural Networks: The Evolution of Technology from Web 1.0 to AI."**\n\nThis outline is designed to be engaging, educational, and structured for readability, suitable for a tech-savvy audience or general readers interested in digital history.\n\n---\n\n### **Blog Title Options**\n*   *Option 1:* From Static to Smart: The Journey from Web 1.0 to Artificial Intelligence\n*   *Option 2:* The Digital Evolution: How We Moved from Reading the Web to Talking to AI\n*   *Option 3:* Web 1.0, 2.0, and the AI Era: A Timeline of Technological Leapfrogging\n\n### **Target Audience**\n*   Tech enthusiasts\n*   Digital marketers\n*   Students and educators\n*   Business leaders looking to understand the current tech landscape\n\n### **Tone**\n*   Informative yet accessible\n*   Nostalgic (for Web 1.0)\n*   Analytical (for Web 2.0)\n*   Forward-looking and slightly specula